# Part 4 — Mitigation

Three techniques, one per ML-pipeline stage:

1. **Pre-processing** — `aif360.Reweighing` to compute per-sample weights, fed into a weighted cross-entropy `Trainer`.
2. **Post-processing** — `fairlearn.ThresholdOptimizer` with `equalized_odds`, plus a Pareto sweep of the constraint tolerance.
3. **Data augmentation** — 3× oversampling of the high-black training cohort.

All three are compared to the Part 1 baseline on a single summary table, and the demographic-parity vs equalized-odds incompatibility is quantified at the end.

In [ ]:
import json, os, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)
from datasets import Dataset

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric
from fairlearn.postprocessing import ThresholdOptimizer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
THRESHOLD = json.load(open('artifacts/chosen_threshold.json'))['threshold']
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128

In [ ]:
train_df = pd.read_parquet('artifacts/train.parquet')
eval_df  = pd.read_parquet('artifacts/eval.parquet')

def mark_groups(df):
    df = df.copy()
    df['is_high_black'] = (df['black']  >= 0.5).astype(int)
    df['is_reference'] = ((df['black'] <  0.1) & (df['white'] >= 0.5)).astype(int)
    return df

train_df = mark_groups(train_df)
eval_df  = mark_groups(eval_df)

print('Train high_black:', int(train_df['is_high_black'].sum()),
      '   reference:',    int(train_df['is_reference'].sum()))
print('Eval  high_black:', int(eval_df['is_high_black'].sum()),
      '   reference:',    int(eval_df['is_reference'].sum()))

### Shared helpers — tokenisation and Trainer setup

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize(batch):
    return tokenizer(batch['comment_text'], truncation=True, max_length=MAX_LEN)

def to_ds(df, cols=None):
    keep = cols or ['comment_text', 'label']
    ds = Dataset.from_pandas(df[keep])
    return ds.map(tokenize, batched=True, remove_columns=['comment_text'])

def training_args(out_dir):
    return TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=3,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.06,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        logging_steps=200,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to='none',
        remove_unused_columns=False,
    )

def compute_metrics(eval_pred):
    logits, y = eval_pred
    p = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(y, p), 'f1_macro': f1_score(y, p, average='macro')}

@torch.no_grad()
def predict_probs(model_dir, texts, batch_size=64):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i+batch_size], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors='pt').to(DEVICE)
        logits = mdl(**enc).logits
        out.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy().tolist())
    return np.array(out)

## Technique 1 — Reweighing (aif360)

For reweighing to have a handle on the bias, the protected attribute must be present in the training set. Rows outside either cohort receive privileged-group weight 1.0 — they do not pull the weights in either direction.

The weighted loss is passed to the Trainer by subclassing `Trainer.compute_loss`. This is the standard pattern for sample-weighted fine-tuning in HuggingFace.

In [ ]:
mask_train = (train_df['is_high_black'] == 1) | (train_df['is_reference'] == 1)
aif_df = train_df[mask_train].copy()
aif_df['group'] = aif_df['is_reference']  # 1 = reference (privileged), 0 = high_black

bld = BinaryLabelDataset(
    df=aif_df[['group', 'label']].rename(columns={'label': 'y'}),
    label_names=['y'], protected_attribute_names=['group'],
    favorable_label=0, unfavorable_label=1,
)

rw = Reweighing(unprivileged_groups=[{'group': 0}], privileged_groups=[{'group': 1}])
bld_rw = rw.fit_transform(bld)
cohort_weights = bld_rw.instance_weights

weights = np.ones(len(train_df), dtype=np.float32)
weights[mask_train.values] = cohort_weights.astype(np.float32)
train_df['sample_weight'] = weights

pd.Series(weights).describe().round(3)

In [ ]:
REWEIGH_DIR = 'distilbert_reweighed'

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        w = inputs.pop('sample_weight').to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
        per_sample = loss_fn(logits, labels)
        loss = (per_sample * w).mean()
        return (loss, outputs) if return_outputs else loss

rw_train = to_ds(train_df.rename(columns={'label': 'label'}),
                 cols=['comment_text', 'label', 'sample_weight'])
rw_eval  = to_ds(eval_df, cols=['comment_text', 'label'])

rw_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
rw_trainer = WeightedTrainer(
    model=rw_model, args=training_args(REWEIGH_DIR),
    train_dataset=rw_train, eval_dataset=rw_eval,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)
rw_trainer.train()
rw_trainer.save_model(REWEIGH_DIR)
tokenizer.save_pretrained(REWEIGH_DIR)

## Technique 2 — Threshold optimisation (fairlearn)

Rather than retrain, hold the baseline model frozen and choose per-cohort thresholds that satisfy `equalized_odds`. `ThresholdOptimizer` wraps a base estimator exposing `predict_proba`; here the 'estimator' is a frozen scorer over the baseline probabilities.

In [ ]:
base_probs = np.load('artifacts/baseline_eval_probs.npy')
base_labels = np.load('artifacts/baseline_eval_labels.npy')

mask_eval = (eval_df['is_high_black'] == 1) | (eval_df['is_reference'] == 1)
X_eval = base_probs[mask_eval.values].reshape(-1, 1)
y_eval = base_labels[mask_eval.values]
groups = eval_df.loc[mask_eval, 'is_reference'].values  # 1=reference, 0=high_black

class ProbScorer:
    """Thin adapter so ThresholdOptimizer can call predict / predict_proba."""
    def fit(self, X, y):
        return self
    def predict(self, X):
        return (X[:, 0] >= 0.5).astype(int)
    def predict_proba(self, X):
        p = X[:, 0]
        return np.stack([1 - p, p], axis=1)

to_opt = ThresholdOptimizer(
    estimator=ProbScorer(),
    constraints='equalized_odds',
    objective='accuracy_score',
    prefit=True,
)
to_opt.fit(X_eval, y_eval, sensitive_features=groups)

preds_to = to_opt.predict(X_eval, sensitive_features=groups)
print('Equalized-odds post-processing accuracy:', round(accuracy_score(y_eval, preds_to), 4))

### Pareto sweep — equal opportunity difference vs overall F1

Sweep a grid of per-group threshold adjustments, starting from the baseline single threshold of 0.4 and nudging the high-black threshold upward. At each point compute overall F1 and the equal-opportunity difference on the full evaluation set.

In [ ]:
all_probs = base_probs
all_labels = base_labels
all_high_black = eval_df['is_high_black'].values
all_reference  = eval_df['is_reference'].values

def eod_for(t_high_black, t_reference, t_rest):
    p = np.where(all_high_black == 1, (all_probs >= t_high_black),
        np.where(all_reference  == 1, (all_probs >= t_reference),
                                       (all_probs >= t_rest))).astype(int)
    def tpr(mask):
        m = (mask == 1) & (all_labels == 1)
        return p[m].mean() if m.sum() else np.nan
    return (tpr(all_high_black) - tpr(all_reference)), p

rows = []
for t_hb in np.arange(0.35, 0.76, 0.025):
    eod, preds = eod_for(t_hb, 0.4, 0.4)
    rows.append({'t_high_black': round(float(t_hb), 3),
                 'equal_opportunity_diff': float(eod),
                 'f1_macro': f1_score(all_labels, preds, average='macro')})
pareto = pd.DataFrame(rows)
pareto.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(pareto['equal_opportunity_diff'].abs(), pareto['f1_macro'], marker='o')
ax.set_xlabel('|Equal-opportunity difference|  (lower = more fair)')
ax.set_ylabel('F1 macro (accuracy proxy)')
ax.set_title('Accuracy–fairness Pareto frontier (post-processing sweep)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Technique 3 — Oversampling

Duplicate every high-black training row 3× (so each original row appears 4× in total). Concatenate with the original training frame, shuffle, retrain from scratch.

In [ ]:
hb_rows = train_df[train_df['is_high_black'] == 1]
augmented = pd.concat([train_df, hb_rows, hb_rows, hb_rows], ignore_index=True)
augmented = augmented.sample(frac=1, random_state=SEED).reset_index(drop=True)
print('Original train rows:', len(train_df), ' augmented:', len(augmented))

In [ ]:
OVER_DIR = 'distilbert_oversampled'
over_train = to_ds(augmented, cols=['comment_text', 'label'])
over_eval  = to_ds(eval_df,   cols=['comment_text', 'label'])

over_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
over_trainer = Trainer(
    model=over_model, args=training_args(OVER_DIR),
    train_dataset=over_train, eval_dataset=over_eval,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)
over_trainer.train()
over_trainer.save_model(OVER_DIR)
tokenizer.save_pretrained(OVER_DIR)

## Comparison across all four models

In [ ]:
def cohort_stats(probs, df, threshold=THRESHOLD):
    preds = (probs >= threshold).astype(int)
    y = df['label'].values
    overall_f1 = f1_score(y, preds, average='macro')
    def fpr(mask):
        m = (mask == 1) & (y == 0)
        return preds[m].mean() if m.sum() else np.nan
    def tpr(mask):
        m = (mask == 1) & (y == 1)
        return preds[m].mean() if m.sum() else np.nan
    def pos_rate(mask):
        m = mask == 1
        return preds[m].mean() if m.sum() else np.nan
    hb = df['is_high_black'].values; rf = df['is_reference'].values
    return {
        'F1_overall':        overall_f1,
        'FPR_high_black':    fpr(hb),
        'FPR_reference':     fpr(rf),
        'StatParityDiff':    pos_rate(rf) - pos_rate(hb),
        'EqualOppDiff':      tpr(rf) - tpr(hb),
    }

baseline_p  = np.load('artifacts/baseline_eval_probs.npy')
reweigh_p   = predict_probs(REWEIGH_DIR, eval_df['comment_text'].tolist())
oversamp_p  = predict_probs(OVER_DIR,    eval_df['comment_text'].tolist())

def thresh_opt_probs_to_preds():
    # Map the ThresholdOptimizer decision back to a probability for rows it covered;
    # non-cohort rows retain the baseline threshold decision at 0.4.
    preds = (baseline_p >= THRESHOLD).astype(int)
    covered = eval_df['is_high_black'].values | eval_df['is_reference'].values
    groups_full = eval_df.loc[covered == 1, 'is_reference'].values
    preds[covered == 1] = to_opt.predict(baseline_p[covered == 1].reshape(-1, 1),
                                         sensitive_features=groups_full)
    return preds

thr_preds = thresh_opt_probs_to_preds()

def stats_from_preds(preds, df):
    y = df['label'].values
    hb = df['is_high_black'].values; rf = df['is_reference'].values
    def fpr(mask):
        m = (mask == 1) & (y == 0)
        return preds[m].mean() if m.sum() else np.nan
    def tpr(mask):
        m = (mask == 1) & (y == 1)
        return preds[m].mean() if m.sum() else np.nan
    def pos_rate(mask):
        m = mask == 1
        return preds[m].mean() if m.sum() else np.nan
    return {
        'F1_overall':     f1_score(y, preds, average='macro'),
        'FPR_high_black': fpr(hb),
        'FPR_reference':  fpr(rf),
        'StatParityDiff': pos_rate(rf) - pos_rate(hb),
        'EqualOppDiff':   tpr(rf) - tpr(hb),
    }

rows = [
    ('baseline',           cohort_stats(baseline_p,  eval_df)),
    ('reweighing',         cohort_stats(reweigh_p,   eval_df)),
    ('threshold_optim',    stats_from_preds(thr_preds, eval_df)),
    ('oversampling',       cohort_stats(oversamp_p,  eval_df)),
]
summary = pd.DataFrame([r[1] for r in rows], index=[r[0] for r in rows]).round(4)
summary

In [ ]:
# Persist probs + pick the best mitigated model for Part 5.
best_name = summary['FPR_high_black'].idxmin()
print('Best model on FPR_high_black:', best_name)

np.save('artifacts/reweighed_probs.npy',    reweigh_p)
np.save('artifacts/oversampled_probs.npy',  oversamp_p)
summary.to_csv('artifacts/part4_summary.csv')

best_dir = {
    'baseline':        'distilbert_baseline',
    'reweighing':      REWEIGH_DIR,
    'threshold_optim': 'distilbert_baseline',
    'oversampling':    OVER_DIR,
}[best_name]

with open('artifacts/best_mitigated.json', 'w') as f:
    json.dump({'name': best_name, 'model_dir': best_dir, 'threshold': THRESHOLD}, f)
print('Pipeline will load:', best_dir)

### Can we satisfy demographic parity AND equalized odds at the same time?

No. The two constraints are only jointly satisfiable when the base rates (prevalence of the positive label) are equal across cohorts — which is essentially never true in content moderation.

In [ ]:
base_hb = eval_df.loc[eval_df['is_high_black'] == 1, 'label'].mean()
base_rf = eval_df.loc[eval_df['is_reference']  == 1, 'label'].mean()
print(f'Base rate (toxic fraction) high_black : {base_hb:.4f}')
print(f'Base rate (toxic fraction) reference  : {base_rf:.4f}')
print(f'Difference                            : {base_hb - base_rf:+.4f}')

**Why the two are incompatible when base rates differ.** Demographic parity demands equal positive prediction rates: `P(pred=1 | group=a) = P(pred=1 | group=b)`. Equalized odds demands equal TPR and equal FPR across groups. Using the law of total probability:

$$P(\hat{Y}=1 \mid A=g) = \mathrm{TPR}_g \cdot \pi_g + \mathrm{FPR}_g \cdot (1 - \pi_g),$$

where $\pi_g$ is the base rate in group $g$. If equalized odds holds ($\mathrm{TPR}_a = \mathrm{TPR}_b$ and $\mathrm{FPR}_a = \mathrm{FPR}_b$) but $\pi_a \neq \pi_b$, the two positive rates must differ — so demographic parity fails. Conversely, forcing demographic parity while base rates differ requires either lowering TPR or raising FPR on one of the groups, which breaks equalized odds.

With the numbers above, equalising positive prediction rates would require the model to flag roughly the *same fraction* of each cohort regardless of the underlying toxicity prevalence — which either silences innocent users in the higher-base-rate cohort or lets real toxicity slip in the lower-base-rate cohort. **Kleinberg, Mullainathan & Raghavan (2016)** prove this formally: no non-trivial classifier can satisfy both calibration and equalized odds unless base rates match or the classifier is perfect. Our run is a concrete instantiation of that impossibility. The engineering answer is to pick the fairness axis that matters most for the harm being caused (here, equal FPR, because over-flagging is the dominant harm) and optimise for that.